In [0]:
import logging

logging.basicConfig(level=logging.INFO, format="%(asctime)s [%(levelname)s] %(message)s")
logger = logging.getLogger(__name__)

# ADF passes these as notebook parameters
dbutils.widgets.text("bronze_path", "")
dbutils.widgets.text("silver_path", "")
dbutils.widgets.text("last_watermark", "")

bronze_path    = dbutils.widgets.get("bronze_path")
silver_path    = dbutils.widgets.get("silver_path")
last_watermark = dbutils.widgets.get("last_watermark")

# Fail fast if required params are missing
if not bronze_path or not silver_path:
    raise ValueError("bronze_path and silver_path must be provided by ADF")

logger.info(f"Bronze path   : {bronze_path}")
logger.info(f"Silver path   : {silver_path}")
logger.info(f"Last watermark: {last_watermark if last_watermark else 'FULL LOAD'}")

2026-04-18 16:54:57,351 [INFO] Bronze path   : abfss://bronze@stnyctaxisri01.dfs.core.windows.net/nyc_taxi/trips/
2026-04-18 16:54:57,353 [INFO] Silver path   : abfss://silver@stnyctaxisri01.dfs.core.windows.net/nyc_taxi/trips/
2026-04-18 16:54:57,354 [INFO] Last watermark: 2026-04-18 21:46:24


In [0]:
import pyspark.sql.functions as F

logger.info(f"Reading bronze data from: {bronze_path}")

try:
    df_bronze = spark.read.format("parquet").load(bronze_path)
    logger.info(f"Bronze read success — row count: {df_bronze.count()}, columns: {df_bronze.columns}")
except Exception as e:
    logger.error(f"Failed to read bronze data: {str(e)}")
    raise

2026-04-18 16:49:06,893 [INFO] Reading bronze data from: abfss://bronze@stnyctaxisri01.dfs.core.windows.net/nyc_taxi/trips/
2026-04-18 16:49:07,668 [INFO] Bronze read success — row count: 30, columns: ['trip_id', 'vendor_id', 'pickup_datetime', 'dropoff_datetime', 'passenger_count', 'trip_distance', 'pickup_longitude', 'pickup_latitude', 'dropoff_longitude', 'dropoff_latitude', 'payment_type', 'fare_amount', 'tip_amount', 'total_amount', 'rate_code', 'store_fwd_flag', 'updated_at', 'created_at', 'year', 'month', 'day']


In [0]:
from pyspark.sql.types import StructType, StructField, IntegerType, StringType, TimestampType, DecimalType, ByteType

# Baseline schema — locked from source
EXPECTED_SCHEMA = {
    "trip_id":           "int",
    "vendor_id":         "string",
    "pickup_datetime":   "timestamp",
    "dropoff_datetime":  "timestamp",
    "passenger_count":   "tinyint",
    "trip_distance":     "decimal(38,18)",
    "pickup_longitude":  "decimal(38,18)",
    "pickup_latitude":   "decimal(38,18)",
    "dropoff_longitude": "decimal(38,18)",
    "dropoff_latitude":  "decimal(38,18)",
    "payment_type":      "string",
    "fare_amount":       "decimal(38,18)",
    "tip_amount":        "decimal(38,18)",
    "total_amount":      "decimal(38,18)",
    "rate_code":         "string",
    "store_fwd_flag":    "string",
    "updated_at":        "timestamp",
    "created_at":        "timestamp"
}

# Partition cols added by ADF — exclude from drift check
PARTITION_COLS = {"year", "month", "day"}

# Actual schema from bronze read
actual_schema = {
    f.name: f.dataType.simpleString()
    for f in df_bronze.schema.fields
    if f.name not in PARTITION_COLS
}

# Check 1 — missing columns (in expected but not in actual)
missing_cols = set(EXPECTED_SCHEMA.keys()) - set(actual_schema.keys())

# Check 2 — new columns (in actual but not in expected)
new_cols = set(actual_schema.keys()) - set(EXPECTED_SCHEMA.keys())

# Check 3 — type mismatches
type_mismatches = {
    col: {"expected": EXPECTED_SCHEMA[col], "actual": actual_schema[col]}
    for col in EXPECTED_SCHEMA.keys() & actual_schema.keys()
    if EXPECTED_SCHEMA[col] != actual_schema[col]
}

# Report
if missing_cols:
    logger.error(f"Schema drift — MISSING columns: {missing_cols}")
if new_cols:
    logger.warning(f"Schema drift — NEW columns detected: {new_cols}")
if type_mismatches:
    logger.error(f"Schema drift — TYPE mismatches: {type_mismatches}")

# Fail pipeline on critical drift
if missing_cols or type_mismatches:
    raise ValueError(
        f"Critical schema drift detected. Missing: {missing_cols}. "
        f"Type mismatches: {type_mismatches}. Pipeline aborted."
    )

# New columns — warn but continue (non-breaking)
if new_cols:
    logger.warning(f"New columns will be ignored in processing: {new_cols}")
    df_bronze = df_bronze.select(list(EXPECTED_SCHEMA.keys()))
    logger.info("DataFrame trimmed to expected schema")

logger.info("Schema validation passed")

2026-04-18 16:49:07,787 [INFO] Schema validation passed


In [0]:
from pyspark.sql.window import Window

logger.info(f"Applying incremental filter — watermark: {last_watermark}")

# Incremental filter
df_filtered = df_bronze.filter(F.col("updated_at") > last_watermark)

logger.info(f"Rows after incremental filter: {df_filtered.count()}")

# Deduplication — keep latest record per trip_id
window = Window.partitionBy("trip_id").orderBy(F.col("updated_at").desc())

df_deduped = df_filtered \
    .withColumn("rn", F.row_number().over(window)) \
    .filter(F.col("rn") == 1) \
    .drop("rn")

logger.info(f"Rows after deduplication: {df_deduped.count()}")

2026-04-18 16:49:08,012 [INFO] Applying incremental filter — watermark: 2026-04-18 21:46:24
2026-04-18 16:49:08,254 [INFO] Rows after incremental filter: 10
2026-04-18 16:49:08,723 [INFO] Rows after deduplication: 10


In [0]:
if df_deduped.count() == 0:
    logger.warning("No new data after filtering — exiting early")
    dbutils.notebook.exit("NO_DATA")

logger.info("Data available — proceeding to silver write")

2026-04-18 16:49:09,259 [INFO] Data available — proceeding to silver write


In [0]:
from delta.tables import DeltaTable

silver_table = "nyc_taxi_prod.silver.trips"

logger.info(f"Writing to silver table: {silver_table}")

if spark.catalog.tableExists(silver_table):
    logger.info("Silver table exists — running MERGE")
    delta_table = DeltaTable.forName(spark, silver_table)
    
    delta_table.alias("t").merge(
        df_deduped.alias("s"),
        "t.trip_id = s.trip_id"
    ).whenMatchedUpdateAll() \
     .whenNotMatchedInsertAll() \
     .execute()
    
    logger.info("MERGE complete")

else:
    logger.info("Silver table does not exist — creating with initial load")
    
    df_deduped.write \
        .format("delta") \
        .mode("overwrite") \
        .option("path", f"{silver_path}") \
        .saveAsTable(silver_table)
    
    logger.info(f"Silver table created: {silver_table}")

2026-04-18 16:49:09,387 [INFO] Writing to silver table: nyc_taxi_prod.silver.trips
2026-04-18 16:49:10,532 [INFO] Silver table exists — running MERGE
2026-04-18 16:49:17,495 [INFO] MERGE complete


In [0]:
%sql
SELECT COUNT(*) as row_count FROM nyc_taxi_prod.silver.trips;

row_count
40


In [0]:
from delta.tables import DeltaTable

logger.info("Running OPTIMIZE on silver table")

spark.sql("""
    OPTIMIZE nyc_taxi_prod.silver.trips
    ZORDER BY (trip_id, pickup_datetime)
""")

logger.info("OPTIMIZE complete")

2026-04-18 16:49:22,554 [INFO] Running OPTIMIZE on silver table
2026-04-18 16:49:25,242 [INFO] OPTIMIZE complete


In [0]:
logger.info("Running VACUUM on silver table")
spark.sql("VACUUM nyc_taxi_prod.silver.trips RETAIN 168 HOURS")
logger.info("VACUUM complete")

In [0]:
import json

max_watermark = df_deduped.agg(F.max("updated_at")).collect()[0][0]
row_count = df_deduped.count()

output = {
    "max_watermark": str(max_watermark),
    "rows_processed": row_count
}

logger.info(f"Notebook output: {output}")
dbutils.notebook.exit(json.dumps(output))